## Workshop


In [1]:
from kafka import KafkaConsumer
import json 

server = "localhost:9092"
topic_name = "rides"

def ride_deserializer(data):
    json_str = data.decode('utf-8')
    return json.loads(json_str)

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset="earliest",
    group_id="rides-console",
    value_deserializer=ride_deserializer,
)

In [17]:
next(consumer)

KeyboardInterrupt: 

## Homework

In [1]:
from models import ride_deserializer
from kafka import KafkaConsumer

server = "localhost:9092"
topic_name = "green-trips"
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset="earliest",
    group_id="rides-console",
    value_deserializer=ride_deserializer,  # json_serializer for test purpose
)

In [2]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [4]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.lpep_pickup_datetime / 1000)
    dropoff_dt = datetime.fromtimestamp(ride.lpep_dropoff_datetime / 1000)
    cur.execute(
        """INSERT INTO processed_events
           (PULocationID, DOLocationID, trip_distance, tip_amount, passenger_count, total_amount, pickup_datetime, dropoff_datetime)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
        (ride.PULocationID, ride.DOLocationID,
         ride.trip_distance, ride.tip_amount, ride.passenger_count, ride.total_amount, pickup_dt, dropoff_dt)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to green-trips and writing to PostgreSQL...


KeyboardInterrupt: 